# Lista 5 — Zadanie 3: Eksploracja modeli encoder-only (20 pkt)

Porównujemy co najmniej **dwa aspekty**:
1. **Różne modele** klasyfikujące (HerBERT vs inny model z Hugging Face)
2. **Parametr `max_length`** — jak długi kontekst wpływa na wyniki

Wyniki zestawiamy w tabeli porównawczej.

In [ ]:
import sys

!{sys.executable} -m pip install -q torch transformers datasets scikit-learn pandas

In [ ]:
import sys
from pathlib import Path

TASK5_DIR = Path("..").resolve()
if str(TASK5_DIR) not in sys.path:
    sys.path.insert(0, str(TASK5_DIR))

import torch
import pandas as pd
from transformers import pipeline

from common.data import load_polemo_test
from common.labels import map_text_to_class
from common.metrics import evaluate_predictions, print_evaluation

## Krok 1: Przygotowanie danych

In [ ]:
examples = load_polemo_test()
sentences = [ex["sentence"] for ex in examples]
y_true = [ex["class"] for ex in examples]

device = 0 if torch.cuda.is_available() else -1
print(f"Próbek: {len(sentences)} | Urządzenie: {'GPU' if device == 0 else 'CPU'}")

## Krok 2: Funkcja pomocnicza do eksperymentów

In [ ]:
def run_encoder_experiment(model_name, max_length=512, batch_size=16):
    """Ładuje model, klasyfikuje dane i zwraca metryki."""
    pipe = pipeline(
        "text-classification",
        model=model_name,
        device=device,
    )

    raw_outputs = pipe(
        sentences,
        batch_size=batch_size,
        truncation=True,
        max_length=max_length,
    )

    y_pred = []
    for output in raw_outputs:
        mapped = map_text_to_class(output["label"])
        y_pred.append(mapped if mapped else "neutral")

    metrics = evaluate_predictions(y_true, y_pred)
    return metrics

## Eksperyment A: Porównanie modeli

| Model | Opis |
|-------|------|
| `Voicelab/herbert-base-cased-sentiment` | HerBERT fine-tuned na sentyment ogólny (recenzje) |
| `bardsai/finance-sentiment-pl-base` | HerBERT fine-tuned na sentyment finansowy (inna domena) |

In [ ]:
MODELS_TO_COMPARE = [
    "Voicelab/herbert-base-cased-sentiment",
    "bardsai/finance-sentiment-pl-base",
]

model_results = []
for model_name in MODELS_TO_COMPARE:
    print(f"\n>>> Uruchamiam: {model_name}")
    try:
        metrics = run_encoder_experiment(model_name, max_length=512)
        print_evaluation(metrics, title=model_name)
        model_results.append({
            "eksperyment": "model",
            "wariant": model_name.split("/")[-1],
            "accuracy": metrics["accuracy"],
            "f1_macro": metrics["f1_macro"],
            "f1_weighted": metrics["f1_weighted"],
        })
    except Exception as e:
        print(f"Błąd dla {model_name}: {e}")
        print("Sprawdź nazwę modelu na https://huggingface.co/models?pipeline_tag=text-classification&language=pl")

## Eksperyment B: Wpływ max_length

Sprawdzamy, czy skracanie kontekstu (128 vs 512 tokenów) zmienia jakość klasyfikacji.

In [ ]:
BASE_MODEL = "Voicelab/herbert-base-cased-sentiment"
MAX_LENGTHS = [128, 256, 512]

length_results = []
for max_len in MAX_LENGTHS:
    print(f"\n>>> max_length = {max_len}")
    metrics = run_encoder_experiment(BASE_MODEL, max_length=max_len)
    print_evaluation(metrics, title=f"max_length={max_len}")
    length_results.append({
        "eksperyment": "max_length",
        "wariant": str(max_len),
        "accuracy": metrics["accuracy"],
        "f1_macro": metrics["f1_macro"],
        "f1_weighted": metrics["f1_weighted"],
    })

## Krok 3: Tabela porównawcza

In [ ]:
comparison_df = pd.DataFrame(model_results + length_results)
comparison_df

## Podsumowanie (do raportu)

- **Model z tej samej domeny** (recenzje) zwykle radzi sobie lepiej niż model z innej domeny (np. finanse).
- **max_length** — zbyt krótki kontekst może obcinać istotne fragmenty długich recenzji.
- Klasa **neutral** jest najtrudniejsza dla obu modeli (niski recall).
- Inne modele PL: [Hugging Face — text-classification (pl)](https://huggingface.co/models?pipeline_tag=text-classification&language=pl).